# 03 — User Segmentation Analysis

**Business question:** Who are your most valuable listeners?

This notebook turns event-level behavior into user-level segments. It identifies high-value listeners, compares power users against casual users, and surfaces when and where different listeners are most engaged.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "master_dataset.csv"
df = pd.read_csv(DATA_PATH)

def first_existing(candidates, required=True):
    for column in candidates:
        if column in df.columns:
            return column
    if required:
        raise KeyError(f"None of these columns exist: {candidates}")
    return None

user_col = first_existing(["user_id"])
content_col = first_existing(["content_id"])
session_col = first_existing(["session_id"])
timestamp_col = first_existing(["timestamp", "started_at", "event_timestamp"], required=False)
format_col = first_existing(["content_format", "format"], required=False)
length_col = first_existing(["content_length_minutes", "duration_minutes"], required=False)
device_col = first_existing(["device_type", "device"], required=False)
subscription_col = first_existing(["subscription_type", "subscription_plan"], required=False)
search_col = first_existing(["search_demand_score", "monthly_searches"], required=False)

if timestamp_col:
    df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors="coerce")
if "play_count" not in df.columns:
    df["play_count"] = 1
if "skip_count" not in df.columns:
    df["skip_count"] = df["skipped"].astype(int) if "skipped" in df.columns else 0
if "is_completed" in df.columns:
    df["is_completed"] = df["is_completed"].astype(bool)

def safe_qcut(series, labels):
    ranked = series.rank(method="first")
    try:
        return pd.qcut(ranked, q=len(labels), labels=labels).astype(int)
    except ValueError:
        return pd.Series(np.ceil(ranked.rank(pct=True) * len(labels)).clip(1, len(labels)).astype(int), index=series.index)

def low_variance_note(column):
    return df[column].nunique(dropna=True) < 2 if column in df.columns else True

print(f"Loaded {df.shape[0]:,} rows × {df.shape[1]:,} columns from {DATA_PATH}")
df.head()

## 1. RFM Segmentation

RFM segmentation adapts classic customer analytics to streaming behavior. Recency captures how recently a user listened, frequency captures repeat behavior, and monetary is represented by total listening duration.

In [ ]:
max_tenure = df["user_tenure_days"].max()
user_rfm = (
    df.groupby(user_col)
    .agg(
        last_tenure_day=("user_tenure_days", "max"),
        frequency=("play_count", "sum"),
        total_duration_minutes=("session_duration_minutes", "sum"),
        is_power_user=("is_power_user", "max"),
    )
    .reset_index()
)
user_rfm["recency"] = max_tenure - user_rfm["last_tenure_day"]
user_rfm["recency_score"] = 4 - safe_qcut(user_rfm["recency"], [1, 2, 3])
user_rfm["frequency_score"] = safe_qcut(user_rfm["frequency"], [1, 2, 3])
user_rfm["monetary_score"] = safe_qcut(user_rfm["total_duration_minutes"], [1, 2, 3])

def label_segment(row):
    if (row["recency_score"], row["frequency_score"], row["monetary_score"]) == (3, 3, 3):
        return "Champions"
    if row["recency_score"] >= 2 and row["frequency_score"] >= 2 and row["monetary_score"] >= 2:
        return "Loyal"
    if row["recency_score"] == 1 and (row["frequency_score"] >= 2 or row["monetary_score"] >= 2):
        return "At Risk"
    if row["recency_score"] == 1 and row["frequency_score"] == 1:
        return "Lost"
    return "New Users"

user_rfm["segment"] = user_rfm.apply(label_segment, axis=1)
fig = px.scatter(
    user_rfm,
    x="frequency",
    y="total_duration_minutes",
    size="recency",
    color="segment",
    hover_data=[user_col, "recency_score", "frequency_score", "monetary_score"],
    title="RFM Bubble Chart: Frequency vs Listening Duration",
    labels={"frequency": "Play Count", "total_duration_minutes": "Total Listening Minutes", "recency": "Days Since Last Session"},
)
fig.show()
user_rfm.sort_values("total_duration_minutes", ascending=False).head(10)

## 2. Power Users vs Casual Listeners

Power listeners often drive a disproportionate share of platform value. Comparing them with casual listeners helps prioritize retention levers, content investment, and personalization depth.

In [ ]:
power_share = df.loc[df["is_power_user"], "session_duration_minutes"].sum() / df["session_duration_minutes"].sum()
comparison = (
    df.groupby("is_power_user")
    .agg(avg_session_duration=("session_duration_minutes", "mean"), avg_skip_count=("skip_count", "mean"), total_playtime=("session_duration_minutes", "sum"), users=(user_col, "nunique"))
    .reset_index()
)

def top_value(group, column):
    if column not in group.columns or group[column].nunique(dropna=True) == 0:
        return "Not available"
    return group[column].mode(dropna=True).iloc[0]

profile_rows = []
for is_power, group in df.groupby("is_power_user"):
    profile_rows.append({
        "is_power_user": is_power,
        "top_genre": top_value(group, "genre"),
        "preferred_time_of_day": top_value(group, "time_of_day"),
    })
comparison = comparison.merge(pd.DataFrame(profile_rows), on="is_power_user", how="left")
print(f"Power listener playtime share: {power_share:.1%}")
comparison

## 3. Peak Listening Heatmap

A daypart-by-weekday heatmap shows when sessions become more valuable. This can guide content drops, notification timing, and campaign scheduling.

In [ ]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
time_order = ["morning", "afternoon", "evening", "night"]
heatmap_data = (
    df.pivot_table(index="time_of_day", columns="day_of_week", values="session_duration_minutes", aggfunc="mean")
    .reindex(index=time_order, columns=day_order)
)
fig = px.imshow(
    heatmap_data,
    text_auto=".1f",
    aspect="auto",
    color_continuous_scale="Blues",
    title="Peak Listening Heatmap: Avg Session Duration by Daypart and Weekday",
    labels={"x": "Day of Week", "y": "Time of Day", "color": "Avg Minutes"},
)
fig.show()
heatmap_data

## 4. Device Breakdown

Device-level engagement reveals whether completion and playtime are concentrated on mobile, desktop, or web experiences. Low-variance device data is still displayed but should be treated as directional.

In [ ]:
if device_col is None:
    raise KeyError("Device column is required for device breakdown.")
device_summary = (
    df.groupby(device_col, dropna=False)
    .agg(total_playtime=("session_duration_minutes", "sum"), completion_rate=("is_completed", "mean"), plays=("play_count", "sum"))
    .reset_index()
)
device_summary["playtime_share"] = device_summary["total_playtime"] / device_summary["total_playtime"].sum()
fig = make_subplots(rows=1, cols=2, subplot_titles=("Playtime Share", "Completion Rate"))
fig.add_trace(go.Bar(x=device_summary[device_col], y=device_summary["playtime_share"], name="Share"), row=1, col=1)
fig.add_trace(go.Bar(x=device_summary[device_col], y=device_summary["completion_rate"], name="Completion"), row=1, col=2)
fig.update_yaxes(tickformat=".0%")
fig.update_layout(title="Device Engagement Breakdown", height=450, showlegend=False)
fig.show()
device_summary

## 💡 Key Finding

Top 20% users (Power Listeners) account for the majority of total playtime in this synthetic dataset. The benchmark target for this portfolio story is 68%; use the computed share above as the live value from the current sample and validate again after regenerating the full dataset.